# Garmin_R1_WholeRun_FullArchive
Labels every run in the archive with its R1 whole-run label.

**Inputs**
- `.fit` files in `garmin_data\DI_CONNECT\Sorted\Run` (named `DDMMYYYY__id.fit`)

**Logic**
1. Load each file, clean speed, resample to 1 Hz
2. Split the run into segments at pace changes
3. Label each segment (Strides, Recovery, Fast/Effort, Z2/Easy) relative to the run's median speed
4. Merge fast segments into bouts and apply rules to give the whole-run label
5. Check the last quarter of the run for strides

**Outputs** (in `garmin_data\Processed`)
- `r1_whole_run_labels.csv`: one row per run with `file_path`, `date`, `r1_whole_run_label`
  - e.g. `23112024__293732718463.fit, 2024-11-23, Z2/Steady State`
- `r1_skipped_files.csv`: files not labelled, with `file_path`, `reason`
  - e.g. `14062018__200005.fit, no segments found`

Join to other label files on `file_path`.

## 1. Setup and configuration

In [ ]:
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from garmin_fit_sdk import Decoder, Stream

# Paths
RUN_DIR = Path(r"C:\dev\UoW\DATAX504\Final Assignment\garmin_data\DI_CONNECT\Sorted\Run")
OUTPUT_DIR = Path(r"C:\dev\UoW\DATAX504\Final Assignment\garmin_data\Processed")
OUTPUT_PATH = OUTPUT_DIR / "r1_whole_run_labels.csv"
SKIPPED_PATH = OUTPUT_DIR / "r1_skipped_files.csv"

# Loading and cleaning
MIN_RECORDS = 50
MAX_SPEED_KMH = 25
MAX_INTERP_GAP_S = 5

# Segmentation
ROLLING_WINDOW = 30
VARIANCE_THRESHOLD_PCTL = 85
MIN_SEGMENT_LENGTH = 60

# Segment types (multiples of run median speed)
STRIDE_SEG_MAX_S = 45
STRIDE_SEG_SPEED_MULT = 1.40
RECOVERY_SPEED_MULT = 0.80
FAST_SPEED_MULT = 1.08

# Whole-run rules
MIN_BOUT_S = 45
TEMPO_MIN_S = 8 * 60
SPLIT_TEMPO_MIN_BOUT_S = 5 * 60
SPLIT_TEMPO_MAX_GAP_S = 2 * 60
TAIL_PATTERN_MAX_BOUT_S = 5 * 60
TAIL_FRACTION = 0.25

# Tail strides
TAIL_STRIDE_SPEED_MULT = 1.20
TAIL_STRIDE_MIN_S = 8
TAIL_STRIDE_MAX_S = 75
TAIL_STRIDE_MIN_COUNT = 3

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Select all run files

In [ ]:
def parse_date_from_filename(fpath):
    date_str = fpath.stem.split("__")[0]
    return datetime.strptime(date_str, "%d%m%Y")


run_files = list(RUN_DIR.glob("*.fit"))
print(f"Total sorted running files: {len(run_files)}")

selected = [(f, parse_date_from_filename(f)) for f in run_files]
selected.sort(key=lambda x: x[1])
print(f"Date range: {selected[0][1].date()} to {selected[-1][1].date()}")
print(f"Processing all {len(selected)} files.")

## 3. Load, clean and resample

In [ ]:
def extract_run(fpath):
    stream = Stream.from_file(str(fpath))
    decoder = Decoder(stream)
    messages, errors = decoder.read()

    if "session_mesgs" not in messages or "record_mesgs" not in messages:
        return None

    rows = []
    for record in messages["record_mesgs"]:
        hr = record.get("heart_rate")
        speed = record.get("enhanced_speed", record.get("speed"))
        timestamp = record.get("timestamp")
        if None not in (hr, speed, timestamp):
            rows.append({"timestamp": timestamp, "heart_rate": hr, "speed": speed})

    if len(rows) < MIN_RECORDS:
        return None

    df = pd.DataFrame(rows).sort_values("timestamp").reset_index(drop=True)
    df["elapsed_s"] = (df["timestamp"] - df["timestamp"].iloc[0]).dt.total_seconds()
    return df


def clean_speed(speed_array, max_kmh=MAX_SPEED_KMH):
    speed_kmh = speed_array * 3.6
    implausible = (speed_kmh > max_kmh) | (speed_kmh < 0)
    cleaned = pd.Series(speed_array.copy())
    cleaned[implausible] = np.nan
    cleaned = cleaned.interpolate(limit=MAX_INTERP_GAP_S)
    return cleaned.bfill().ffill().to_numpy()


def resample_to_1s(df):
    df = df.drop(columns=["timestamp"])
    for col in ["heart_rate", "speed"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    indexed = df.set_index("elapsed_s")
    full_index = np.arange(0, int(indexed.index.max()) + 1, 1)
    resampled = indexed.reindex(indexed.index.union(full_index)).sort_index()
    resampled = resampled.interpolate(method="linear", limit_direction="both")
    resampled = resampled.reindex(full_index)
    resampled["elapsed_s"] = resampled.index
    return resampled.reset_index(drop=True)


def prepare_run(fpath):
    df = extract_run(fpath)
    if df is None:
        return None
    df["speed"] = clean_speed(df["speed"].to_numpy())
    return resample_to_1s(df)

## 4. Segmentation and segment types

In [ ]:
def detect_segments(df, rolling_window=ROLLING_WINDOW, pctl=VARIANCE_THRESHOLD_PCTL, min_len=MIN_SEGMENT_LENGTH):
    speed = df["speed"].to_numpy()
    n = len(speed)
    rolling_var = pd.Series(speed).rolling(rolling_window, center=True, min_periods=1).var().fillna(0.0).to_numpy()
    threshold = np.percentile(rolling_var, pctl)
    is_transition = rolling_var > threshold

    boundaries = [0]
    current_state = is_transition[0]
    for index in range(1, n):
        if is_transition[index] != current_state:
            boundaries.append(index)
            current_state = is_transition[index]
    boundaries.append(n)

    segments = []
    segment_start = boundaries[0]
    for boundary in boundaries[1:]:
        if boundary - segment_start >= min_len:
            segments.append((segment_start, boundary))
            segment_start = boundary
    if segment_start < n:
        segments.append((segment_start, n))
    return segments


def classify_segment_type(seg_speed, seg_length_s, run_median_speed):
    avg_speed = seg_speed.mean()
    if seg_length_s < STRIDE_SEG_MAX_S and avg_speed > run_median_speed * STRIDE_SEG_SPEED_MULT:
        return "Strides"
    if avg_speed < run_median_speed * RECOVERY_SPEED_MULT:
        return "Recovery"
    if avg_speed > run_median_speed * FAST_SPEED_MULT:
        return "Fast/Effort"
    return "Z2/Easy"


def get_fast_bouts(run_segments):
    fast_bouts = []
    current = None
    for _, segment in run_segments.sort_values("start_s").iterrows():
        if segment["type_label"] != "Fast/Effort":
            current = None
            continue
        if current is not None and segment["start_s"] <= current["end_s"] + 1:
            old_duration = current["duration_s"]
            added_duration = segment["length_s"]
            total_duration = old_duration + added_duration
            current["avg_speed"] = (current["avg_speed"] * old_duration + segment["avg_speed"] * added_duration) / total_duration
            current["end_s"] = segment["end_s"]
            current["duration_s"] = total_duration
        else:
            current = {"start_s": segment["start_s"], "end_s": segment["end_s"],
                       "duration_s": segment["length_s"], "avg_speed": segment["avg_speed"]}
            fast_bouts.append(current)
    return fast_bouts

## 5. Whole-run label

In [ ]:
def detect_tail_strides(df, run_median_speed, tail_fraction=TAIL_FRACTION,
                        speed_multiplier=TAIL_STRIDE_SPEED_MULT,
                        min_stride_s=TAIL_STRIDE_MIN_S, max_stride_s=TAIL_STRIDE_MAX_S,
                        min_stride_count=TAIL_STRIDE_MIN_COUNT):
    speed = df["speed"].to_numpy()
    elapsed = df["elapsed_s"].to_numpy()
    tail_start_s = elapsed[-1] * (1 - tail_fraction)
    tail_indices = np.where(elapsed >= tail_start_s)[0]
    if len(tail_indices) == 0:
        return []
    smooth_speed = pd.Series(speed).rolling(5, center=True, min_periods=1).median().to_numpy()
    stride_threshold = run_median_speed * speed_multiplier
    above_threshold = smooth_speed >= stride_threshold
    above_threshold[:tail_indices[0]] = False

    strides = []
    index = tail_indices[0]
    while index < len(above_threshold):
        if not above_threshold[index]:
            index += 1
            continue
        start_idx = index
        while index < len(above_threshold) and above_threshold[index]:
            index += 1
        end_idx = index
        duration_s = end_idx - start_idx
        if min_stride_s <= duration_s <= max_stride_s:
            strides.append({"start_idx": start_idx, "end_idx": end_idx, "start_s": elapsed[start_idx],
                            "end_s": elapsed[end_idx - 1], "duration_s": duration_s,
                            "avg_speed": np.nanmean(speed[start_idx:end_idx])})
        index += 1
    return strides if len(strides) >= min_stride_count else []


def summarize_run_type(run_segments):
    fast_bouts = get_fast_bouts(run_segments)
    meaningful = [b for b in fast_bouts if b["duration_s"] >= MIN_BOUT_S]
    durations = [b["duration_s"] for b in meaningful]
    run_median_speed = run_segments["run_median_speed"].iloc[0]
    clear_interval_bouts = [b for b in meaningful if b["avg_speed"] >= run_median_speed * FAST_SPEED_MULT]
    interval_count = len(clear_interval_bouts)

    run_end_s = run_segments["end_s"].max()
    final_quarter_start = run_end_s * (1 - TAIL_FRACTION)
    tail_stride_pattern = (
        len(meaningful) >= 2
        and all(b["start_s"] >= final_quarter_start for b in meaningful)
        and max(durations) <= TAIL_PATTERN_MAX_BOUT_S
    )

    if tail_stride_pattern:
        base = "Z2/Steady State with Strides"
    elif interval_count >= 3:
        base = "Structured Intervals"
    elif len(meaningful) == 1:
        base = "Tempo Run" if meaningful[0]["duration_s"] >= TEMPO_MIN_S else "Z2/Steady State"
    elif len(meaningful) == 2:
        recovery_duration = meaningful[1]["start_s"] - meaningful[0]["end_s"]
        split_tempo = (min(meaningful[0]["duration_s"], meaningful[1]["duration_s"]) >= SPLIT_TEMPO_MIN_BOUT_S
                       and recovery_duration <= SPLIT_TEMPO_MAX_GAP_S)
        if split_tempo:
            base = "Tempo Run"
        elif interval_count == 2:
            base = "Structured Intervals"
        else:
            base = "Z2/Steady State"
    else:
        base = "Z2/Steady State"

    if (run_segments["type_label"] == "Strides").any() and "with Strides" not in base:
        base += " with Strides"
    return base


def label_run(df):
    speed = df["speed"].to_numpy()
    moving_speed = speed[speed > 0]
    run_median_speed = np.median(moving_speed) if len(moving_speed) else speed.mean()

    segments = detect_segments(df)
    if len(segments) == 0:
        return None

    rows = []
    for start, end in segments:
        seg_speed = speed[start:end]
        seg_length_s = end - start
        rows.append({
            "start_s": start, "end_s": end, "length_s": seg_length_s,
            "avg_speed": seg_speed.mean(), "run_median_speed": run_median_speed,
            "type_label": classify_segment_type(seg_speed, seg_length_s, run_median_speed),
        })
    run_segments = pd.DataFrame(rows)

    label = summarize_run_type(run_segments)
    strides = detect_tail_strides(df, run_median_speed)
    if label == "Z2/Steady State" and strides:
        label = "Z2/Steady State with Strides"

    return {"label": label, "segments": run_segments, "strides": strides}

## 6. Label all files and save

In [ ]:
run_rows = []
skipped = []

for i, (fpath, date) in enumerate(selected):
    if i % 100 == 0:
        print(f"...processing {i}/{len(selected)} files - {len(run_rows)} runs labelled so far")

    try:
        df = prepare_run(fpath)
        if df is None:
            skipped.append({"file_path": fpath.name, "reason": "missing messages or fewer than MIN_RECORDS usable records"})
            continue

        result = label_run(df)
        if result is None:
            skipped.append({"file_path": fpath.name, "reason": "no segments found"})
            continue

        run_rows.append({"file_path": fpath.name, "date": date, "r1_whole_run_label": result["label"]})

    except Exception as exc:
        skipped.append({"file_path": fpath.name, "reason": f"error: {exc}"})

print(f"\nDone. Labelled {len(run_rows)} runs successfully, {len(skipped)} skipped.")

r1_df = pd.DataFrame(run_rows)
r1_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(r1_df)} whole-run labels to {OUTPUT_PATH}")

skipped_df = pd.DataFrame(skipped, columns=["file_path", "reason"])
skipped_df.to_csv(SKIPPED_PATH, index=False)
print(f"Saved {len(skipped_df)} skipped files to {SKIPPED_PATH}")

print("\nLabel counts:")
print(r1_df["r1_whole_run_label"].value_counts())